In [ ]:
#pip install python-dotenv semantic-kernel[azure]

In [ ]:
import os
import json
from typing import List
from dataclasses import dataclass, asdict
from semantic_kernel.functions.kernel_function_decorator import kernel_function

@dataclass
class FlightModel:
    Id: int
    Airline: str
    Destination: str
    DepartureDate: str
    Price: float
    IsBooked: bool = False


class FlightBookingPlugin:
    FILE_PATH = "flights.json"

    def __init__(self):
        self.flights: List[FlightModel] = self.load_flights_from_file()


    # Create a plugin function with kernel function attributes
    @kernel_function(description="Searches for available flights based on the destination and departure date in the format YYYY-MM-DD")
    def search_flights(self, destination, departure_date):
        # Filter flights based on destination
        matching_flights = [
            flight for flight in self.flights
            if flight.Destination.lower() == destination.lower() and flight.DepartureDate == departure_date
        ]
        return matching_flights



    # Create a kernel function to book flights
    @kernel_function(description="Books a flight based on the flight ID provided")
    def book_flight(self, flight_id):
        # Add logic to book a flight
        flight = next((f for f in self.flights if f.Id == flight_id), None)
            
        if flight is None:
            return "Flight not found. Please provide a valid flight ID."

        if flight.IsBooked:
            return "You've already booked this flight."

        flight.IsBooked = True
        self.save_flights_to_file()

        return (
            f"Flight booked successfully! Airline: {flight.Airline}, "
            f"Destination: {flight.Destination}, Departure: {flight.DepartureDate}, "
            f"Price: ${flight.Price}."
        )


    def save_flights_to_file(self):
        with open(self.FILE_PATH, 'w', encoding='utf-8') as f:
            json.dump([asdict(flight) for flight in self.flights], f, indent=4)

    def load_flights_from_file(self) -> List[FlightModel]:
        if os.path.exists(self.FILE_PATH):
            with open(self.FILE_PATH, 'r', encoding='utf-8') as f:
                data = json.load(f)
                return [FlightModel(**item) for item in data]

        raise FileNotFoundError(f"The file '{self.FILE_PATH}' was not found. Please provide a valid flights.json file.")

In [ ]:
import os
import asyncio
from dotenv import load_dotenv
from semantic_kernel import Kernel
from semantic_kernel.contents.chat_history import ChatHistory
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion, AzureChatPromptExecutionSettings
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior
from semantic_kernel.functions.kernel_arguments import KernelArguments
#from flight_booking_plugin import FlightBookingPlugin

async def main():

    load_dotenv()
    # Set your values in the .env file
    api_key = os.getenv("AZURE_OPENAI_KEY")
    endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
    deployment_name = os.getenv("DEPLOYMENT_NAME")

    kernel = Kernel()
    chat_completion = AzureChatCompletion(
        api_key=api_key,
        endpoint=endpoint,
        deployment_name=deployment_name
    )
    kernel.add_service(chat_completion)

    # Add the plugin to the kernel
    kernel.add_plugin(FlightBookingPlugin(), "flight_booking_plugin")


    # Configure function choice behavior
    settings=AzureChatPromptExecutionSettings(
        function_choice_behavior=FunctionChoiceBehavior.Auto(filters={"included_functions": ["search_flights"]}),
    )

    chat_history = ChatHistory()
    chat_history.add_system_message("The year is 2025 and the current month is January")

    async def get_reply():
        reply = await chat_completion.get_chat_message_content(
            chat_history=chat_history,
            kernel=kernel,
            settings=settings
        )
        print("Assistant:", reply)
        chat_history.add_assistant_message(str(reply))

    def get_input():
        user_input = input("User: ")
        if user_input.strip() != "":
            chat_history.add_user_message(user_input)
        return user_input

    def add_user_message(msg: str):
        print(f"User: {msg}")
        chat_history.add_user_message(msg)

    chat_history.add_user_message("Find me a flight to Tokyo on the 19")
    await get_reply()
    get_input()
    await get_reply()

if __name__ == "__main__":
        asyncio.run(main())